# Conversion of TheWell GSDR -> PDEBench DR format

1) Identical to DPOT's conversion, only need to transpose the data order to match PDEBench DR conventions.
2) Data normalization, corresponding scripts included.
3) Inference using TheWell GSDR pipeline.

## Note
I am using 'bubbles' variant for TheWell GSDR.

# Data normalization for original TheWell GSDR

In [4]:
import numpy as np
!python MORPH/scripts/data_normalization_revin_GSDR.py


[GSDR] Importing test data...
Shape of GSDR2D data (20, 1001, 1, 128, 128, 1, 2)
Reshape of MHD data (20, 1001, 2, 1, 1, 128, 128)
Running ReVIN (Numpy array of shape must be (N, T, F, C, D, H, W))...
Normalize dataset shape (20, 1001, 2, 1, 1, 128, 128)
Round-trip max abs error: 8.941e-08
Normed Data in Raw shape (20, 1001, 1, 128, 128, 1, 2)
Raw shape of GSDR2d: (20, 1001, 128, 128, 2)
[GSDR2D] Saved file: gray_scott_reaction_diffusion_bubbles_F_0.098_k_0.057.hdf5, chunks=20, shape(s) A=(20, 1001, 128, 128), B=(20, 1001, 128, 128)
Normalized GSDR data saved under: /Volumes/T7/MORPH_Processed/normalized_revin/2dgrayscottdr_thewell


# Inference for original TheWell GSDR

In [9]:
!python MORPH/scripts/infer_MORPH.py \
    --model_choice FM \
    --model_size S \
    --checkpoint morph-S-FM-max_ar1_ep225.pth \
    --test_dataset GSDR2D \
    --ar_order 1 \
    --rollout_horizon 10 \
    --batch_size 1 \
    --test_sample 0 \
    --max_ar_order 1

DEBUG: Project Root is set to: /Users/sky/Git/MORPH/MORPH
Number of CUDA devices = 0
No CUDA devices available 

→ Selected Batch size for GSDR2D is 1
→ Selected Model MORPH-FM-S for Dataset GSDR2D
→ Loading test dataset...
[GSDR] Importing test data...
→ [GSDR2D] Shape of test data: (20, 1001, 1, 128, 128, 1, 2)
→[GSDR2D] Dataset preparation...
Images(N,T,F,C,D,H,W): torch.Size([20000, 1, 2, 1, 1, 128, 128]), Targets(N,F,C,D,H,W): torch.Size([20000, 2, 1, 1, 128, 128])
→ Length dataloader: 20000
→ NUMBER OF PARAMETERS OF THE ViT (in millions): 32.8
TESTING MODEL:
ViT3DRegression(
  (patch_embedding): HybridPatchEmbedding3D(
    (conv_features): ConvOperator(
      (input_proj): Conv3d(3, 8, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
      (conv_stack): Sequential(
        (0): Conv3d(8, 8, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): LeakyReLU(negative_slope=0.2, inplace=True)
      )
    )
    (projection): Linear(in_features=4096, ou

# Conversion of TheWell GSDR -> PDEBench DR

Just stack ['A','B'] of 't0_fields' of TheWell GSDR together to get 'data' for PDEBench DR format.

In [6]:
import h5py

pdb_dr = '/Volumes/T7/PDEBench/2D/diffusion-reaction/dr_sliced_0_99.h5'

with h5py.File(pdb_dr,'r') as f:
    print(f.keys())
    print(f['0000'].keys())
    print(f['0000']['data'])
    print(f['0000']['grid']['x'])
    print(f['0000']['grid']['y'])
    print(f['0000']['grid']['t'])

# 101 timesteps, 128x128, 2 chemical species concentration

<KeysViewHDF5 ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013', '0014', '0015', '0016', '0017', '0018', '0019', '0020', '0021', '0022', '0023', '0024', '0025', '0026', '0027', '0028', '0029', '0030', '0031', '0032', '0033', '0034', '0035', '0036', '0037', '0038', '0039', '0040', '0041', '0042', '0043', '0044', '0045', '0046', '0047', '0048', '0049', '0050', '0051', '0052', '0053', '0054', '0055', '0056', '0057', '0058', '0059', '0060', '0061', '0062', '0063', '0064', '0065', '0066', '0067', '0068', '0069', '0070', '0071', '0072', '0073', '0074', '0075', '0076', '0077', '0078', '0079', '0080', '0081', '0082', '0083', '0084', '0085', '0086', '0087', '0088', '0089', '0090', '0091', '0092', '0093', '0094', '0095', '0096', '0097', '0098', '0099']>
<KeysViewHDF5 ['data', 'grid']>
<HDF5 dataset "data": shape (101, 128, 128, 2), type "<f4">
<HDF5 dataset "x": shape (128,), type "<f4">
<HDF5 dataset "y": shape (128,), type "<f4">
<H

In [3]:
import h5py

# dr_pdb_revin = '/Volumes/T7/MORPH_Processed/normalized_revin/DR2d_data_pdebench/test/dr_sliced_0_99_test.h5'
#
# with h5py.File(dr_pdb_revin,'r') as f:
#     print(f.keys())
#     print(f['0090'].keys())
#     print(f['0090']['data'])

print()

gsdr_thewell = '/Volumes/T7/MORPH_Processed/2dgrayscottdr_thewell/thewell/test/gray_scott_reaction_diffusion_bubbles_F_0.098_k_0.057.hdf5'

with h5py.File(gsdr_thewell,'r') as f:
    print(f.keys())
    # print(f['scalars'].keys())
    # print(f['dimensions'].keys())
    print(f['t0_fields'].keys())
    print(f['t0_fields']['A'])
    # print(f['t1_fields'].keys())
    # print(f['t2_fields'].keys())

    print()

# gsdr_thewell_revin = '/Volumes/T7/MORPH_Processed/normalized_revin/2dgrayscottdr_thewell/test/gray_scott_reaction_diffusion_bubbles_F_0.098_k_0.057.hdf5'
#
# with h5py.File(gsdr_thewell_revin,'r') as f:
#     print(f.keys())
#     print(f['t0_fields'].keys())
#     print(f['t0_fields']['A'])


<KeysViewHDF5 ['boundary_conditions', 'dimensions', 'scalars', 't0_fields', 't1_fields', 't2_fields']>
<KeysViewHDF5 ['A', 'B']>
<HDF5 dataset "A": shape (20, 1001, 128, 128), type "<f4">



In [28]:
#Convert GSDR to PDB
#Only need t0_fields, stack them

import os
import numpy as np

output_path = '/Volumes/T7/MORPH_Processed/gsdr_converted_pdbformat.h5'
input_path = '/Volumes/T7/MORPH_Processed/2dgrayscottdr_thewell/test/gray_scott_reaction_diffusion_bubbles_F_0.098_k_0.057.hdf5'

os.makedirs(os.path.dirname(output_path), exist_ok=True)

with h5py.File(input_path, 'r') as f_in, h5py.File(output_path, 'w') as f_out:
    A = f_in['t0_fields']['A']
    B = f_in['t0_fields']['B']

    num_sims = A.shape[0]

    print(num_sims)

    for i in range(num_sims):
        all_data = np.stack([A[i],B[i]],axis=-1)

        group_name = f'{i:04d}'

        grp = f_out.create_group(group_name)
        grp.create_dataset('data', data=all_data)

print('Done')

20
Done


# Sanity check

In [30]:
import h5py

output_path = '/Volumes/T7/MORPH_Processed/gsdr_converted_pdbformat.h5'

with h5py.File(output_path, 'r') as f:
    print(f.keys())
    print(f['0000'].keys())
    print(f['0000']['data'])

<KeysViewHDF5 ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013', '0014', '0015', '0016', '0017', '0018', '0019']>
<KeysViewHDF5 ['data']>
<HDF5 dataset "data": shape (1001, 128, 128, 2), type "<f4">


# Data Normalization for converted TheWell GSDR

In [3]:
!python MORPH/scripts/data_normalization_revin_DR_thewell.py

Spliting the file into train/val/test...
Found 1 HDF5 files in '/Volumes/T7/MORPH_Processed/DR2d_data_pdebench'
Processing file: gsdr_bubbles.h5...

>>> Keys in gsdr_bubbles.h5: ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013', '0014', '0015', '0016', '0017', '0018', '0019']
[DR] Saved 0 → /Volumes/T7/MORPH_Processed/DR2d_data_pdebench/train/gsdr_bubbles_train.h5
[DR] Saved 0 → /Volumes/T7/MORPH_Processed/DR2d_data_pdebench/val/gsdr_bubbles_val.h5
[DR] Saved 20 → /Volumes/T7/MORPH_Processed/DR2d_data_pdebench/test/gsdr_bubbles_test.h5
[DR] Importing test data...
Shape of inflated DR data (20, 1001, 1, 128, 128, 1, 2)
Transposed DR data (20, 1001, 2, 1, 1, 128, 128)
Running ReVIN (Numpy array of shape must be (N, T, F, C, D, H, W))...
Normalize dataset shape (20, 1001, 2, 1, 1, 128, 128)
Denormalized dataset shape (20, 1001, 2, 1, 1, 128, 128)
DR RevIN round-trip OK
Normed dataset in raw shape (20, 1001, 128, 128, 2)
Saved n

# Inference for converted TheWell GSDR

In [31]:
#Converted GSDR -> PDB DR, results same as original
!python MORPH/scripts/infer_MORPH.py \
    --model_choice FM \
    --model_size S \
    --checkpoint morph-S-FM-max_ar1_ep225.pth \
    --test_dataset DR \
    --ar_order 1 \
    --rollout_horizon 10 \
    --batch_size 1 \
    --test_sample 0 \
    --max_ar_order 1

DEBUG: Project Root is set to: /Users/sky/Git/MORPH/MORPH
Number of CUDA devices = 0
No CUDA devices available 

→ Selected Batch size for DR is 1
→ Selected Model MORPH-FM-S for Dataset DR
→ Loading test dataset...
[DR] Importing test data...
→ [DR] Shape of test data: (20, 1001, 1, 128, 128, 1, 2)
→[DR] Dataset preparation...
Images(N,T,F,C,D,H,W): torch.Size([20000, 1, 2, 1, 1, 128, 128]), Targets(N,F,C,D,H,W): torch.Size([20000, 2, 1, 1, 128, 128])
→ Length dataloader: 20000
→ NUMBER OF PARAMETERS OF THE ViT (in millions): 32.8
TESTING MODEL:
ViT3DRegression(
  (patch_embedding): HybridPatchEmbedding3D(
    (conv_features): ConvOperator(
      (input_proj): Conv3d(3, 8, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
      (conv_stack): Sequential(
        (0): Conv3d(8, 8, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): LeakyReLU(negative_slope=0.2, inplace=True)
      )
    )
    (projection): Linear(in_features=4096, out_features=512, bi

# Results
TheWell GSDR -> PDEBench DR format:

RMSE: 1.12400, MAE: 0.73339, MSE: 1.26337 VRMSE: 0.51430, NRMSE: 0.32407

Original, unchanged TheWell GSDR results:

RMSE: 1.12400, MAE: 0.73339, MSE: 1.26337 VRMSE: 0.51430, NRMSE: 0.32407


# Observations/Conclusion
As expected, the results of the converted TheWell GSDR does not differ from the original. The conversion works as intended.